# Using Reasoning Models

So far in the examples, we have used a `gpt-4o` model. Models like `gpt-4o` and `gemini-1.5-flash` are great at following instructions, so you can have relatively detailed instructions in the selector prompt for the team and the system messages for each agent to guide their behavior.

However, if you are using a reasoning model like `o3-mini`, you will need to keep the "selector prompt" and" system messages" as simple and to the point as possible. This is because the reasoning models are already good at coming up with their own instructions given the `context` provided to them.

This also means that we don’t need a planning agent to break down the task anymore, since the `SelectorGroupChat` that uses a reasoning model can do that on its own.

In the following example, we will use o3-mini as the model for the agents and the team, and we will not use a planning agent. Also, we are keeping the selector prompt and system messages as simple as possible.

#### ***Reference URL***

https://microsoft.github.io/autogen/stable//user-guide/agentchat-user-guide/selector-group-chat.html#using-reasoning-models

In [ ]:
model_client = OpenAIChatCompletionClient(model="o3-mini")

web_search_agent = AssistantAgent(
    "WebSearchAgent",
    description="An agent for searching information on the web.",
    tools=[search_web_tool],
    model_client=model_client,
    system_message="""Use web search tool to find information.""",
)

data_analyst_agent = AssistantAgent(
    "DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""Use tool to perform calculation. If you have not seen the data, ask for it.""",
)

user_proxy_agent = UserProxyAgent(
    "UserProxyAgent",
    description="A user to approve or disapprove tasks.",
)

selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
When the task is complete, let the user approve or disapprove the task.
"""

team = SelectorGroupChat(
    [web_search_agent, data_analyst_agent, user_proxy_agent],
    model_client=model_client,
    termination_condition=termination,  # Use the same termination condition as before.
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,
)
